In [2]:
!pip install gymnasium

   ---------------------------------------- 0.0/953.9 kB ? eta -:--:--
   - ------------------------------------- 41.0/953.9 kB 991.0 kB/s eta 0:00:01
   ---------------- ----------------------- 389.1/953.9 kB 4.9 MB/s eta 0:00:01
   ---------------------------------------  931.8/953.9 kB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 953.9/953.9 kB 7.5 MB/s eta 0:00:00


In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

# Simple DQN Model
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )

    def forward(self, x):
        return self.net(x)

# Simple Prioritized Replay Buffer
class PERBuffer:
    def __init__(self):
        self.memory = []
        self.priority = []

    def add(self, experience, error):
        self.memory.append(experience)
        self.priority.append(abs(error) + 1e-5)

    def sample(self):
        p = np.array(self.priority) / sum(self.priority)
        index = np.random.choice(len(self.memory), p=p)
        return self.memory[index], index

    def update(self, index, error):
        self.priority[index] = abs(error) + 1e-5

# Create environment
env = gym.make("CartPole-v1")

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

policy = DQN(state_size, action_size)
target = DQN(state_size, action_size)
target.load_state_dict(policy.state_dict())

optimizer = optim.Adam(policy.parameters(), lr=0.001)
memory = PERBuffer()

# Take one step in environment
state, _ = env.reset()
action = env.action_space.sample()

next_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated

# Calculate initial TD Error
with torch.no_grad():
    current_q = policy(torch.FloatTensor(state))[action]
    next_q = target(torch.FloatTensor(next_state)).max()
    target_q = reward + 0.99 * next_q * (1 - int(done))
    error = target_q - current_q

# Store experience
memory.add((state, action, reward, next_state, done), error.item())

# Sample experience
(sample_state, sample_action, sample_reward,
 sample_next_state, sample_done), index = memory.sample()

# Train network
pred = policy(torch.FloatTensor(sample_state))[sample_action]

with torch.no_grad():
    target_value = sample_reward + 0.99 * target(torch.FloatTensor(sample_next_state)).max() * (1 - int(sample_done))

loss = nn.MSELoss()(pred, target_value)

optimizer.zero_grad()
loss.backward()
optimizer.step()

# Update priority
new_error = abs((target_value - pred).item())
memory.update(index, new_error)

print("Training step completed successfully!")

Training step completed successfully!
